TITLE:            Calculating the global temporal averages of climate variables under SSP245, SAI-1.0, SAI-1.5.  
PROJECT:          "Climate analogs under SAI"  
AUTHORS:          Ruoyu Chen  
COLLABORATORS:    
DATA INPUT:       5 (06 to 10) ensemble files from climate variable folders (TSMX, PRECT, SST...) for each of SSP245, SAI-1.0, and SAI-1.5 scenarios.  
                   
DATA OUTPUT:      .nc file of the temporal and ensemble average over a selected period for climate variables (TSMX, PRECT, SST...) for the 3 climate scenarios. Saved in `"/mnt/research/nasabio/data/climate/L1/present"` for present period and `"/mnt/research/nasabio/data/climate/L1/future"` for future period.  
DATE:             Initiated: 5 June 2026; last run: ____  
OVERVIEW:         Root path is: "/mnt/research/nasabio/data/climate/L1".   
REQUIRES:         So far 30 ensemble files: ensembles 006-010 from TSMX, PRECT, SST in SSP245, SAI-1.0, SAI-1.5 folders.  
NOTES:            Code file #1.

In [2]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns
import geopandas as gpd
import earthpy as et
import xarray as xr
# Spatial subsetting of netcdf files
import regionmask
import glob

# Plotting options
sns.set(font_scale=1.3)
sns.set_style("white")


import warnings
warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*",
    category=UserWarning
)

import earthpy as et

/mnt/home/f0113797/.conda/envs/testgeo4/lib/python3.14/site-packages/earthpy/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


### `create_avg_ncfile` ReadMe: 
This function takes ensemble members and takes the average of each dataset over a time frame, and then takes an average over all the ensembles. This function also automatically saves the output at the specified `output_path` and `file_name` as an .nc file.  
The parameters are: `members, start_year, end_year, input_path, unqiue_title, output_path, file_name`.  
* `members`: `list` of `Strings`. This is a list of the ensemble members you want to take the average of. Careful with inputs, because this is directly combined with the `input_path` and `unqiue_title` to get the full path to the ensemble members.
* `start_year`: `int`. The first year you want to include in your average.
* `end_year`: `int`. The last year you want to include in your average.
* `input_path`: `String` of the path from where you can access the ensemble files. 
* `unqiue_title`: `String` of the names of the ensemble files. This is included to get the file names of the ensembles. Make sure all your ensemble files start with this name to use the code.
* `output_path`: `String` path file to where you want to save the output file.
* `file_name`: `String` of the name of the .nc output file.  
  

Example code:  
```
members_no_008 = ['006', '007', '009', '010']
create_avg_ncfile(members, 2015, 2034, "/mnt/research/nasabio/data/climate/L1/SSP245/PRECT", "precip_indices_", "/mnt/research/nasabio/data/climate/L1/present", "SSP245_PRECT.nc")
```
This slices each ensemble member from 2015 to 2034 and takes the average, and then takes another average of the ensembles '006', '007', '009', '010', and then saves it to `L1/present`.

In [14]:
#Function pulling everything together:

#input_path should be the full path to the .nc files. AND the specific file names for the members. 
def create_avg_ncfile(members, start_year, end_year, input_path, unqiue_title, output_path, file_name):
    path_files = []
    time_sliced_files = []
    # Get the list of all files and directories

    list_of_files = []
    for each in members:
        if (each.endswith("p95_threshold.nc")):
            list_of_files.remove(each)
        else: 
            path = input_path + "/" + unqiue_title + each + ".nc"
            file = xr.open_dataset(path)
            path_files.append(file)
    
    #For every ensemble, collapse the "year" dimension by averaging all values across the years at every lat and lon. 
    for each in path_files[:]: 
        sliced = each.sel(year = slice(start_year, end_year))
        each_ensemble_avg = sliced.mean(dim='year')
        time_sliced_files.append(each_ensemble_avg)

    # As a check of the code, let's look at if the first ensemble came out correctly. 
    # sliced1 = path_files[0].sel(year = slice(start_year, end_year)) #Data over the specified years. 
    # print(sliced1['CDD'].values[:, 2, 0]) #Here is the first value of the 3rd row for each of the 20 years. 
    # print((sliced1['CDD'].values[:, 2, 0]).mean()) #And it's mean. 
    # print(time_sliced_files[0]['CDD'].values[2, 0]) #And compare that with the 3rd row, 1st column of the averaged data. 
    
    concat_files_avg=xr.concat(time_sliced_files, dim='ensemble')
    concat_files_avg = concat_files_avg.assign_coords({'ensemble' : members})
    concat_files_avg = concat_files_avg.mean(dim='ensemble')

    concat_files_avg.attrs["clim_reference"] = concat_files_avg.attrs["scenario"] + " " + concat_files_avg.attrs["variable"] + " " + str(start_year) + "-" + str(end_year)
    
    concat_files_avg.to_netcdf(output_path + "/" + file_name)
    
    return concat_files_avg

# NEED TO RERUN CREATING AVERAGE FILES TO UPDATE CLIMATE_REFERENCE INFORMATION IN DATA ATTRIBUTES. 

### Creating averaged files  
This is implementing `create_avg_ncfile` to create temporal averages in present (2015-2034) and future (2050-2069) to create .nc files, and is saved in "/mnt/research/nasabio/data/climate/L1/present" for present period and "/mnt/research/nasabio/data/climate/L1/future" for future period. 

In [21]:
#Creating all future datasets 2050-2069

members = ['006', '007', '008', '009', '010']
members_no_008 = ['006', '007', '009', '010']

# create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p5/PRECT", "precip_indices_", "/mnt/research/nasabio/data/climate/L1/future", "ARISE_SAI_1p5_PRECT.nc")
# create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p5/SST/marine_heatwave", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/future", "ARISE_SAI_1p5_SST_marine_heatwave.nc")
# create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p5/TSMX/extreme_high", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/future", "ARISE_SAI_1p5_TSMX_extreme_high.nc")

# create_avg_ncfile(members_no_008, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p0/PRECT", "precip_indices_", "/mnt/research/nasabio/data/climate/L1/future", "ARISE_SAI_1p0_PRECT.nc")
# create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p0/SST/marine_heatwave", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/future", "ARISE_SAI_1p0_SST_marine_heatwave.nc")
# create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p0/TSMX/extreme_high", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/future", "ARISE_SAI_1p0_TSMX_extreme_high.nc")

# create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/SSP245/PRECT", "precip_indices_", "/mnt/research/nasabio/data/climate/L1/future", "SSP245_PRECT.nc")
# create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/SSP245/SST/marine_heatwave", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/future", "SSP245_SST_marine_heatwave.nc")
# create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/SSP245/TSMX/extreme_high", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/future", "SSP245_TSMX_extreme_high.nc")


# #Creating all present datasets in 2015-2034 for SSP245
# create_avg_ncfile(members, 2015, 2034, "/mnt/research/nasabio/data/climate/L1/SSP245/PRECT", "precip_indices_", "/mnt/research/nasabio/data/climate/L1/present", "SSP245_PRECT.nc")
# create_avg_ncfile(members, 2015, 2034, "/mnt/research/nasabio/data/climate/L1/SSP245/SST/marine_heatwave", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/present", "SSP245_SST_marine_heatwave.nc")
create_avg_ncfile(members, 2015, 2034, "/mnt/research/nasabio/data/climate/L1/SSP245/TSMX/extreme_high", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/present", "SSP245_TSMX_extreme_high.nc")


<xarray.Dataset> Size: 2MB
Dimensions:         (lat: 192, lon: 288)
Coordinates:
  * lat             (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon             (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
Data variables:
    annual_mean     (lat, lon) float32 221kB -46.48 -46.48 -46.48 ... nan nan
    frequency       (lat, lon) float64 442kB 5.68 5.68 5.68 5.68 ... nan nan nan
    duration        (lat, lon) float64 442kB 4.057 4.057 4.057 ... nan nan nan
    mean_intensity  (lat, lon) float32 221kB -36.89 -36.89 -36.89 ... nan nan
    max_intensity   (lat, lon) float32 221kB -21.67 -21.67 -21.67 ... nan nan
Attributes:
    scenario:               SSP245
    variable:               TSMX
    member:                 006
    event_type:             extreme_high
    season:                 annual
    clim_reference:         SSP245 TSMX 2015-2034
    consec_days_threshold:  3
    extreme_direction:      above
    quantile:               0.9
    land_mask:              Natural Earth 110m
    threshold_method:       pooled individual members, linear detrend

## Used for testing, can safely ignore below. 

In [109]:
#Used for testing.

climate_scenario = "ARISE_SAI_1p5"
climate_variable = "TSMX"
start_year = 2035
end_year = 2069

path_files = []
time_sliced_files = []
path = "/mnt/research/nasabio/data/climate/L1/" + climate_scenario + "/" + climate_variable
list_of_files = glob.glob(path + '/**/*.nc', recursive=True) #Stores the names of all files in the directory in a List


for each in list_of_files[:]: 
    file = xr.open_dataset(each)
    path_files.append(file)      #Opens each file in the folder and adds it to path_files list

for each in path_files[:]:
    sliced = each.sel(year = slice(start_year, end_year))   #From each dataset in the path_file, slice it by years.
    each_ensemble_avg = sliced.mean(dim='year')             #Then get the mean of the values at each location and collapse time dimension
    time_sliced_files.append(each_ensemble_avg)             #Add each file to time_sliced_files. 



concat_files_avg=xr.concat(time_sliced_files, dim='ensemble').mean(dim='ensemble')

concat_files_avg

#.concat first stacks all ensembles along a new dimension, ensemble. 
#.mean then takes the average of all the values along that new dimension, and collapses the ensemble dimension. 

# x = concat_files_avg["annual_mean"]
# x.plot()



#save_path = "/mnt/research/nasabio/data/climate/L1/" + climate_scenario + "/ensemble_avg/"
#file_name = "testing_example_temp1.nc"

save_path = "/mnt/home/f0113797/Documents/testing.nc"

# # #file_name = "EXAMPLE_" + climate_scenario + "_avg_" + climate_variable + "_values_" + str(start_year) + "-" + str(end_year)+ ".nc"
concat_files_avg.to_netcdf(save_path)

concat_files_avg

# return concat_files_avg


<xarray.Dataset> Size: 2MB
Dimensions:         (lat: 192, lon: 288)
Coordinates:
  * lat             (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon             (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
Data variables:
    annual_mean     (lat, lon) float32 221kB -45.94 -45.94 -45.94 ... nan nan
    frequency       (lat, lon) float64 442kB 6.276 6.276 6.276 ... nan nan nan
    duration        (lat, lon) float64 442kB 4.278 4.278 4.278 ... nan nan nan
    mean_intensity  (lat, lon) float32 221kB -38.2 -38.2 -38.2 ... nan nan nan
    max_intensity   (lat, lon) float32 221kB -22.13 -22.13 -22.13 ... nan nan
Attributes:
    scenario:               ARISE_SAI_1p5
    variable:               TSMX
    member:                 008
    event_type:             extreme_high
    season:                 annual
    clim_reference:         SSP245 2015-2034
    consec_days_threshold:  3
    extreme_direction:      above
    quantile:               0.9
    land_mask:              Natural Earth 110m
    threshold_method:       pooled individual members, linear detrend

In [22]:
y1 = xr.open_dataset("/mnt/research/nasabio/data/climate/L1/SSP245/TSMX/extreme_high/extreme_indices_010.nc")
z1 = xr.open_dataset("/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p0/TSMX/extreme_high/extreme_indices_010.nc")

print(y1.attrs["clim_reference"])
print(z1.attrs["clim_reference"])



y = create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/SSP245/TSMX/extreme_high", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/future", "TEST_SSP245_TSMX_extreme_high.nc")
z = create_avg_ncfile(members, 2050, 2069, "/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p0/TSMX/extreme_high", "extreme_indices_", "/mnt/research/nasabio/data/climate/L1/future", "TEST_ARISE_SAI_1p0_TSMX_extreme_high.nc")

SSP245 2015-2034
SSP245 2015-2034


In [16]:
y

<xarray.Dataset> Size: 2MB
Dimensions:         (lat: 192, lon: 288)
Coordinates:
  * lat             (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon             (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
Data variables:
    annual_mean     (lat, lon) float32 221kB -44.62 -44.62 -44.62 ... nan nan
    frequency       (lat, lon) float64 442kB 9.32 9.32 9.32 9.32 ... nan nan nan
    duration        (lat, lon) float64 442kB 4.6 4.6 4.6 4.6 ... nan nan nan nan
    mean_intensity  (lat, lon) float32 221kB -34.57 -34.57 -34.57 ... nan nan
    max_intensity   (lat, lon) float32 221kB -16.35 -16.35 -16.35 ... nan nan
Attributes:
    scenario:               SSP245
    variable:               TSMX
    member:                 006
    event_type:             extreme_high
    season:                 annual
    clim_reference:         SSP245 TSMX 2050-2069
    consec_days_threshold:  3
    extreme_direction:      above
    quantile:               0.9
    land_mask:              Natural Earth 110m
    threshold_method:       pooled individual members, linear detrend

In [17]:
z

<xarray.Dataset> Size: 2MB
Dimensions:         (lat: 192, lon: 288)
Coordinates:
  * lat             (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon             (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
Data variables:
    annual_mean     (lat, lon) float32 221kB -46.85 -46.85 -46.85 ... nan nan
    frequency       (lat, lon) float64 442kB 4.56 4.56 4.56 4.56 ... nan nan nan
    duration        (lat, lon) float64 442kB 4.261 4.261 4.259 ... nan nan nan
    mean_intensity  (lat, lon) float32 221kB -39.57 -39.57 -39.58 ... nan nan
    max_intensity   (lat, lon) float32 221kB -27.94 -27.94 -27.94 ... nan nan
Attributes:
    scenario:               ARISE_SAI_1p0
    variable:               TSMX
    member:                 006
    event_type:             extreme_high
    season:                 annual
    clim_reference:         ARISE_SAI_1p0 TSMX 2050-2069
    consec_days_threshold:  3
    extreme_direction:      above
    quantile:               0.9
    land_mask:              Natural Earth 110m
    threshold_method:       pooled individual members, linear detrend